  ```
  GET  /                           embedded WebUI (enabled by default with a config)
  GET  /health
  GET  /v1/models
  GET  /v1/recipes
  POST /v1/models/load             available with --ui-management
  POST /v1/models/unload           available with --ui-management
  POST /v1/ui/upload               temporary browser upload for WebUI requests
  POST /v1/ui/models/install       background package download/preparation
  POST /v1/ui/models/install/stop  stop one active package download
  POST /v1/ui/models/clean-partial remove abandoned package staging files
  POST /v1/ui/models/delete        remove one installed package precision
  GET  /v1/ui/models-root          current and binary-local default models folders
  POST /v1/ui/models-root          select a models folder (empty path restores default)
  POST /v1/ui/browse-directories   list local folders for the native folder picker
  GET  /v1/ui/models/install-status[?id=<package>]
  GET  /v1/ui/models/package-sizes package sizes from metadata-only checks
  GET  /v1/audio/voices?model=<id>
  POST /v1/audio/speech
  POST /v1/audio/speech/live?model=<id>[&return_text=true]
       raw PCM in a chunked body, speech audio deltas as SSE on the same connection;
       return_text adds the text the model writes as text deltas
  POST /v1/audio/transcriptions
       fields: file, model, language, prompt, stream
  POST /v1/batches/transcriptions
       fields: repeated file, model, language, prompt, options
  POST /v1/audio/alignments
       fields: file, model, text, language
       OpenAI-style streaming: speech stream_format=sse|audio, transcription stream=true
  POST /v1/audio/transcriptions/live?model=<id>
       raw PCM in a chunked body, transcript deltas as SSE on the same connection
  POST /v1/tasks/run
  POST /v1/tasks/batch
  POST /v1/tasks/stream
       events and result in one JSON body, or as SSE with stream_format=sse
  POST /v1/recipes/run
  ```

In [27]:
import httpx
import json
import pydub
BASE_URL = "http://localhost:8919"

response = httpx.post(
    BASE_URL + "/v1/audio/speech",
    content=json.dumps({
        'model': 'supertonic-3',
        'input': "Здравствуйте",
        'language': 'ru'
    })
)

pydub.AudioSegment(response.content)

In [10]:

response = httpx.get(BASE_URL + "/health")
json.loads(response.text)

{'status': 'ok',
 'backend': 'cuda',
 'models': 1,
 'ui': True,
 'ui_management': False,
 'version': 'dev',
 'git': 'a536990',
 'git_date': '2026-10-27'}

In [11]:
response = httpx.get(BASE_URL + "/v1/models")
json.loads(response.text)

{'object': 'list',
 'data': [{'id': 'supertonic-3',
   'object': 'model',
   'owned_by': 'engine',
   'family': 'supertonic',
   'task': 'tts',
   'mode': 'offline',
   'device': 0,
   'loaded': False,
   'configured': True,
   'path': 'models/Supertonic-3-GGUF/'}]}

In [12]:
response = httpx.get(BASE_URL + "/v1/recipes")
json.loads(response.text)

{'object': 'list',
 'data': [{'id': 'any-tts',
   'description': 'Generate speech with any offline TTS model, then clone a reference voice with Tone Color VC',
   'roles': [{'name': 'tts', 'task': 'tts', 'mode': 'offline'},
    {'name': 'converter', 'task': 'vc', 'mode': 'offline'}]},
  {'id': 'cross-language-dubbing',
   'description': 'Translate speech with an ASR translation model and dub it with a target-language TTS model',
   'roles': [{'name': 'translator', 'task': 'asr', 'mode': 'offline'},
    {'name': 'tts', 'task': 'tts', 'mode': 'offline'}]}]}